# Community Notes 陣営反応分析（unzip_data 版）

Drive の `基礎プロジェクト/unzip_data/` から解凍済み TSV を直接コピーして使う実験版です。
安定版は `colab_full_run.ipynb` を使用してください。

上から順にセルを実行してください。
「★ 設定」セルのパラメータを変えることで、実行時間やトピックを調整できます。

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ チャンク実行設定 (全データを少量ずつ分けて処理)
# ====================================================
# 全データを1回で処理すると時間がかかりすぎるため、
# 「ファイル2個 × 行50万」を1チャンクとして分割実行する設計。
#
# ▼ 使い方
#   1. FILE_CHUNK_INDEX, ROW_CHUNK_INDEX を変えてノート全体を再実行
#   2. 各実行で出力が bursts_f{F}_r{R}.csv のように自動で別名保存される
#   3. 全チャンクを回し終えたら最後の「マージ」セルを実行 → *_all.csv が出来る
#
# ▼ 網羅の目安 (8ファイルの場合)
#   FILE_CHUNK_INDEX=0..3 (files[0:2], [2:4], [4:6], [6:8])
#   ROW_CHUNK_INDEX=0,1,2,... (データが尽きるまで。「Loaded 0 rows」が出たらそのファイル群は完了)

FILES_PER_CHUNK  = 2        # 1チャンクで読むファイル数 (変更非推奨)
ROWS_PER_CHUNK   = 500_000  # 1チャンクで読む行数 (変更非推奨)
FILE_CHUNK_INDEX = 0        # ← 0, 1, 2, 3
ROW_CHUNK_INDEX  = 0        # ← 0から順に増やす

# --- 自動導出 (変更不要) ---
FILE_OFFSET   = FILE_CHUNK_INDEX * FILES_PER_CHUNK
SKIP_ROWS     = ROW_CHUNK_INDEX  * ROWS_PER_CHUNK
CHUNK_SUFFIX  = f'_f{FILE_CHUNK_INDEX}_r{ROW_CHUNK_INDEX}'
# 後方互換 (既存セルで使用)
RATINGS_FILES = FILES_PER_CHUNK
NROWS         = ROWS_PER_CHUNK

# --- Drive のパス ---
DRIVE_DATA_UNZIPPED = '/content/drive/Shareddrives/基礎プロジェクト/unzip_data'

# --- 結果の保存先 ---
# Shared Drives は読み取り専用の場合があるため、MyDrive に保存
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results'

# --- 分析対象トピック ---
# キーワードを自由に追加・削除・変更できます
# トピックを減らすと実行が速くなります
TOPICS = {
    'vaccine_covid': [
        'vaccine', 'covid', 'pandemic', 'mask', 'booster',
        'pfizer', 'moderna', 'antivax', 'lockdown', 'fauci',
    ],
    'israel_palestine': [
        'israel', 'palestine', 'gaza', 'hamas', 'idf',
        'netanyahu', 'ceasefire', 'zionist', 'hezbollah',
    ],
    'trump': [
        'trump', 'maga', 'indictment', 'mar-a-lago',
        'january 6', 'j6', 'impeach',
    ],
    'immigration': [
        'immigration', 'border', 'migrant', 'asylum',
        'deportation', 'illegal alien', 'refugee', 'caravan',
    ],
    'gun_control': [
        'gun control', 'second amendment', '2nd amendment',
        'shooting', 'nra', 'firearm', 'gun violence',
    ],
    'ALL_POLITICAL': [
        'trump', 'biden', 'democrat', 'republican', 'gop',
        'congress', 'senate', 'election', 'vote', 'ballot',
        'liberal', 'conservative', 'immigration', 'border',
        'abortion', 'gun control', 'vaccine', 'covid',
        'climate change', 'supreme court',
        'israel', 'palestine', 'gaza', 'ukraine', 'russia',
        'woke', 'dei', 'transgender', 'lgbtq', 'censorship',
        'government', 'policy', 'partisan',
    ],
}

# =============================================
# ★ 分析パラメータ（上級者向け）
# =============================================
# 通常はデフォルト値で問題ありません。
# 分析の感度や閾値を細かく調整したい場合に変更してください。

# --- 前処理: Polarity 計算 ---
POLARITY_FIRST_N = 50   # ← 推奨: 30〜100

# --- 前処理: ノートのフィルタリング ---
MIN_RATING_COUNT = 20    # ← 推奨: 10〜50

# --- 前処理: トピック別分析のフィルタ ---
MIN_RATINGS_TOPIC = 10   # ← 推奨: 5〜30

# --- バースト検出 ---
BURST_SPEED_MULTIPLIER = 3.0  # ← 推奨: 2.0〜5.0
BURST_MIN_COUNT = 5      # ← 推奨: 3〜10

# --- バースト分類: TypeA/TypeB 閾値 ---
BURST_THRESHOLD = None   # ← None（自動）または 0.0〜1.0 程度の数値

# --- 特徴量: トレンド計算の最小評価数 ---
TREND_MIN_EVALS = 4      # ← 推奨: 3〜10

# --- ターゲット抽出: 品質スコア上位 % ---
TARGET_TOP_PERCENT = 25  # ← 推奨: 10〜50

# ====================================================
print(f'▶ チャンク: FILE_CHUNK_INDEX={FILE_CHUNK_INDEX}, ROW_CHUNK_INDEX={ROW_CHUNK_INDEX}')
print(f'  対象ファイル: files[{FILE_OFFSET}:{FILE_OFFSET+FILES_PER_CHUNK}]')
print(f'  行範囲:       [{SKIP_ROWS:,} 〜 {SKIP_ROWS+ROWS_PER_CHUNK:,})')
print(f'  出力サフィックス: {CHUNK_SUFFIX}')
print(f'  例: data/processed/bursts{CHUNK_SUFFIX}.csv')
print()
print(f'save dir: {SAVE_DIR}')
print(f'topics: {list(TOPICS.keys())}')
print()
print('--- 分析パラメータ ---')
print(f'polarity_first_n:       {POLARITY_FIRST_N}')
print(f'min_rating_count:       {MIN_RATING_COUNT}')
print(f'min_ratings_topic:      {MIN_RATINGS_TOPIC}')
print(f'burst_speed_multiplier: {BURST_SPEED_MULTIPLIER}')
print(f'burst_min_count:        {BURST_MIN_COUNT}')
print(f'burst_threshold:        {BURST_THRESHOLD if BURST_THRESHOLD is not None else "自動（中央値）"}')
print(f'trend_min_evals:        {TREND_MIN_EVALS}')
print(f'target_top_percent:     {TARGET_TOP_PERCENT}%')


---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, json

if os.path.exists(DRIVE_DATA_UNZIPPED):
    print('OK: unzip_data フォルダが見つかりました')
    for d in sorted(os.listdir(DRIVE_DATA_UNZIPPED)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA_UNZIPPED} が見つかりません')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('共有ドライブ一覧:')
        for d in os.listdir(sd): print(f'  {d}')
    print('マイドライブ:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]: print(f'  {d}')

In [ ]:
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pip install -q statsmodels

In [ ]:
# data/raw/ にデータを準備（unzip_data から cp、あればスキップ）
# FILE_OFFSET を考慮して対象の ratings ファイル群だけを置く
raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

def _copy_if_missing(src_path, dst_path):
    if os.path.exists(dst_path):
        print(f'  {os.path.basename(dst_path)} already exists, skip')
        return
    print(f'  Copying {os.path.basename(src_path)} ...')
    subprocess.run(['cp', src_path, dst_path], check=True)

# --- ratings: 現在のチャンク分だけコピー、それ以外は削除 ---
ratings_folder = os.path.join(DRIVE_DATA_UNZIPPED, 'RatingData')
if os.path.exists(ratings_folder):
    all_tsvs = sorted([f for f in os.listdir(ratings_folder) if f.startswith('ratings-') and f.endswith('.tsv')])
    chunk_tsvs = all_tsvs[FILE_OFFSET:FILE_OFFSET + FILES_PER_CHUNK]
    if not chunk_tsvs:
        raise RuntimeError(f'FILE_OFFSET={FILE_OFFSET} が ratings tsv 数 {len(all_tsvs)} を超えています')
    needed = set(chunk_tsvs)
    # 余分な ratings tsv を削除
    for f in sorted(os.listdir(raw_dir)):
        if f.startswith('ratings-') and f.endswith('.tsv') and f not in needed:
            print(f'  Removing {f} (not in current chunk)')
            os.remove(os.path.join(raw_dir, f))
    for tsv in chunk_tsvs:
        _copy_if_missing(os.path.join(ratings_folder, tsv), os.path.join(raw_dir, tsv))
else:
    print(f'WARNING: {ratings_folder} not found')

# --- notes, history: あればスキップ ---
other_targets = [
    ('Notes data', 'notes-'),
    ('Note status history data', 'noteStatusHistory-'),
]
for subfolder, prefix in other_targets:
    folder = os.path.join(DRIVE_DATA_UNZIPPED, subfolder)
    if not os.path.exists(folder):
        print(f'WARNING: {folder} not found, skip')
        continue
    for f in sorted(os.listdir(folder)):
        if f.startswith(prefix) and f.endswith('.tsv'):
            _copy_if_missing(os.path.join(folder, f), os.path.join(raw_dir, f))

print()
print('=== data/raw/ ===')
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.tsv'):
        size = os.path.getsize(os.path.join(raw_dir, f)) / (1024**3)
        print(f'  {f}  ({size:.1f} GB)')


---
## 1. 全トピックでパイプライン実行

In [ ]:
nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_pipeline.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --file-offset {FILE_OFFSET} \
    --skip-rows {SKIP_ROWS} \
    --chunk-suffix {CHUNK_SUFFIX} \
    --polarity-first-n {POLARITY_FIRST_N} \
    --min-rating-count {MIN_RATING_COUNT} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS} \
    --target-top-percent {TARGET_TOP_PERCENT}


---
## 2. トピック別比較

In [ ]:
with open('/tmp/topics.json', 'w') as f:
    json.dump(TOPICS, f)

nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_by_topic.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --file-offset {FILE_OFFSET} \
    --skip-rows {SKIP_ROWS} \
    --chunk-suffix {CHUNK_SUFFIX} \
    --min-ratings {MIN_RATINGS_TOPIC} \
    --topics-json /tmp/topics.json \
    --polarity-first-n {POLARITY_FIRST_N} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS}


---
## 3. 結果の確認

In [ ]:
import pandas as pd

df = pd.read_csv(f'data/processed/topic_comparison{CHUNK_SUFFIX}.csv')
print(f'chunk: {CHUNK_SUFFIX}')
display(df)


In [ ]:
targets = pd.read_csv(f'data/processed/target_notes{CHUNK_SUFFIX}.csv')
print(f'chunk: {CHUNK_SUFFIX}  ターゲットノート数: {len(targets)}')
display(targets.head(20))


In [ ]:
burst_path = f'data/processed/bursts{CHUNK_SUFFIX}.csv'
if os.path.exists(burst_path):
    bursts = pd.read_csv(burst_path)
    print(f'chunk: {CHUNK_SUFFIX}  バースト数: {len(bursts)}')
    print(bursts['burst_type'].value_counts())
else:
    print(f'{burst_path} が存在しません (バースト検出 0 件の可能性)')


---
## 4. 全チャンクの結果をマージ（全チャンクを回し終えたら実行）

`data/processed/*_f*_r*.csv` を統合して `*_all.csv` を作成します。


In [ ]:
!python scripts/merge_chunks.py

# 統合後のプレビュー
for name in ['topic_comparison_all.csv', 'target_notes_all.csv', 'bursts_all.csv']:
    path = f'data/processed/{name}'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'\n▼ {name}  ({len(df):,} rows)')
        display(df.head(10))


---
## 5. 結果を Drive に保存


In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
for f in os.listdir('data/processed'):
    src = os.path.join('data/processed', f)
    if f.endswith('.csv') or f.endswith('.png'):
        subprocess.run(['cp', src, SAVE_DIR])
print(f'Done! Results saved to: {SAVE_DIR}')